# AgentCore Policy Lab
## Mastering Amazon Bedrock AgentCore | Pumping Code

---

## 🎯 What You'll Build

In this lab you will build a **deterministic authorization layer** around an Insurance Underwriting AI Agent using **Amazon Bedrock AgentCore Policy** and **Cedar policies**.

By the end of this lab, you will have:
- ✅ Deployed 3 Lambda functions as tool backends
- ✅ Created an AgentCore Gateway with OAuth authentication
- ✅ Run an agent with unrestricted tool access
- ✅ Created and attached a Policy Engine
- ✅ Observed **Default Deny** (empty policy engine blocks everything)
- ✅ Written Cedar policies and tested **ALLOW** and **DENY** scenarios
- ✅ Used **NL2Cedar** to generate policies from natural language

---

## 🏗️ Architecture

```
┌──────────────────┐
│    AI Agent      │  ← Local Strands Agent (Claude Haiku 4.5)
└────────┬─────────┘
         │  MCP Tool Call
         ▼
┌──────────────────┐
│ AgentCore Gateway│  ← OAuth2 Auth (Cognito)
│ + Policy Engine  │  ← Cedar Policy Enforcement
└────────┬─────────┘
         │  ALLOW or DENY
         ▼
┌──────────────────┐
│  Lambda Targets  │  ← Application | Risk Model | Approval tools
└──────────────────┘
```

---

## ✅ Prerequisites
- AWS CLI configured with appropriate credentials
- Python 3.10+
- Access to AWS Lambda, Cognito, and Bedrock
- Bedrock access enabled for: `us.anthropic.claude-haiku-4-5-20251001-v1:0`

This lab runs TypeScript on the Deno kernel. Pick the **Deno** kernel in the top right.

---
# Part 1: Environment Setup

In [ ]:
// Dependencies are pinned in the project's deno.json and cached by ./setup.sh; there is no install
// step. (The Python lab installed bedrock-agentcore-starter-toolkit, boto3, strands-agents,
// strands-agents-tools and requests here.)
import { sh } from "../shared/notebook.ts";
await sh("deno", ["--version"]);

In [ ]:
Deno.env.set("AWS_REGION", "us-east-1");

// APPROACH A: Use credentials
// Deno.env.set("AWS_ACCESS_KEY_ID", "your_access_key");
// Deno.env.set("AWS_SECRET_ACCESS_KEY", "your_secret_key");
// Deno.env.set("AWS_SESSION_TOKEN", "your_session_token");

// APPROACH B: Use AWS SSO profile
// Deno.env.set("AWS_PROFILE", "your_profile");
// Remove any existing credential env vars to force profile usage
// for (const key of ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]) {
//   Deno.env.delete(key);
// }

Deno.env.set("AWS_REGION", "us-east-1");

console.log("✅ AWS Profile set. Please restart kernel and run all cells.");

In [ ]:
import { GetCallerIdentityCommand, STSClient } from "@aws-sdk/client-sts";
import { loadEnv, state, writeFile } from "../shared/notebook.ts";

await loadEnv();

// Verify AWS credentials and region
const region = Deno.env.get("AWS_REGION") ?? "us-east-1";

try {
  const identity = await new STSClient({ region }).send(new GetCallerIdentityCommand({}));
  console.log("✅ AWS Credentials Verified");
  console.log(`   Account: ${identity.Account}`);
  console.log(`   ARN:     ${identity.Arn}`);
  console.log(`   Region:  ${region}`);
} catch (e) {
  console.log(`❌ AWS Credentials Error: ${e}`);
  console.log("   Please configure: aws configure");
}

---
# Part 2: Deploy Infrastructure

We'll deploy **3 Lambda functions** as insurance underwriting tools:

| Tool | Description | Key Parameter |
|------|-------------|---------------|
| `ApplicationTool` | Creates insurance applications | `coverage_amount`, `applicant_region` |
| `RiskModelTool` | Invokes external risk scoring | `API_classification`, `data_governance_approval` |
| `ApprovalTool` | Approves underwriting decisions | `claim_amount`, `risk_level` |

These are simplified mock implementations for demo purposes.

In [ ]:
import JSZip from "jszip";
import {
  CreateFunctionCommand,
  GetFunctionCommand,
  LambdaClient,
  ResourceConflictException,
  UpdateFunctionCodeCommand,
} from "@aws-sdk/client-lambda";
import {
  AttachRolePolicyCommand,
  CreateRoleCommand,
  EntityAlreadyExistsException,
  GetRoleCommand,
  IAMClient,
} from "@aws-sdk/client-iam";
import { isRoleNotReadyError } from "../toolkit/runtime.ts";

const lambdaClient = new LambdaClient({ region });
const iamClient = new IAMClient({ region });

const sleep = (ms: number) => new Promise((resolve) => setTimeout(resolve, ms));

// ── Helper: retry while a brand new IAM role is not assumable yet ─────────
// The Python lab only sleeps for 10 seconds after creating the role, which is not always enough
// for Lambda to accept it. The toolkit already knows what that failure looks like, so reuse it
// rather than letting a propagation delay fail the lab.
async function withRoleRetry<T>(operation: () => Promise<T>, maxRetries = 5): Promise<T> {
  for (let attempt = 0;; attempt++) {
    try {
      return await operation();
    } catch (error) {
      if (!isRoleNotReadyError(error) || attempt === maxRetries) throw error;
      const wait = Math.min(5_000 * 2 ** attempt, 15_000);
      console.log(`   ⏳ IAM role not ready yet, retrying in ${wait / 1000}s...`);
      await sleep(wait);
    }
  }
}

// ── Helper: create or get Lambda execution role ──────────────────────────
async function createLambdaRole(roleName: string): Promise<string> {
  const trustPolicy = JSON.stringify({
    Version: "2012-10-17",
    Statement: [{
      Effect: "Allow",
      Principal: { Service: "lambda.amazonaws.com" },
      Action: "sts:AssumeRole",
    }],
  });
  try {
    const role = await iamClient.send(
      new CreateRoleCommand({
        RoleName: roleName,
        AssumeRolePolicyDocument: trustPolicy,
        Description: "Lambda execution role for AgentCore Policy Lab",
      }),
    );
    await iamClient.send(
      new AttachRolePolicyCommand({
        RoleName: roleName,
        PolicyArn: "arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
      }),
    );
    await sleep(10_000); // IAM propagation
    return role.Role!.Arn!;
  } catch (error) {
    if (!(error instanceof EntityAlreadyExistsException)) throw error;
    const existing = await iamClient.send(new GetRoleCommand({ RoleName: roleName }));
    return existing.Role!.Arn!;
  }
}

// ── Helper: package and deploy a Lambda ──────────────────────────────────
async function deployLambda(name: string, handlerCode: string, roleArn: string): Promise<string> {
  const zip = new JSZip();
  zip.file("index.py", handlerCode);
  const zipBytes = await zip.generateAsync({ type: "uint8array" });

  try {
    const fn = await withRoleRetry(() =>
      lambdaClient.send(
        new CreateFunctionCommand({
          FunctionName: name,
          Runtime: "python3.12",
          Role: roleArn,
          Handler: "index.handler",
          Code: { ZipFile: zipBytes },
          Timeout: 30,
          Description: `AgentCore Policy Lab - ${name}`,
        }),
      )
    );
    console.log(`  ✅ Created: ${name}`);
    return fn.FunctionArn!;
  } catch (error) {
    if (!(error instanceof ResourceConflictException)) throw error;
    await lambdaClient.send(
      new UpdateFunctionCodeCommand({ FunctionName: name, ZipFile: zipBytes }),
    );
    const fn = await lambdaClient.send(new GetFunctionCommand({ FunctionName: name }));
    console.log(`  ♻️  Updated: ${name}`);
    return fn.Configuration!.FunctionArn!;
  }
}

// ── Lambda handler code ───────────────────────────────────────────────────
// These three handlers stay Python, verbatim from the Python lab: they are the payload this
// notebook ships to the Lambda python3.12 runtime, not part of this repo's own toolchain.
const APPLICATION_TOOL_CODE = `
import json

def handler(event, context):
    body = json.loads(event.get("body", "{}")) if isinstance(event.get("body"), str) else event
    region = body.get("applicant_region", "UNKNOWN")
    amount = body.get("coverage_amount", 0)
    app_id = f"APP-{region}-{int(amount/1000)}K"
    return {
        "statusCode": 200,
        "body": json.dumps({
            "application_id": app_id,
            "status": "CREATED",
            "message": f"Insurance application created for {region} with \${amount:,.0f} coverage"
        })
    }
`;

const RISK_MODEL_TOOL_CODE = `
import json

def handler(event, context):
    body = json.loads(event.get("body", "{}")) if isinstance(event.get("body"), str) else event
    api_class = body.get("API_classification", "unknown")
    dg_approval = body.get("data_governance_approval", False)
    risk_score = 0.25 if (api_class == "public" and dg_approval) else 0.75
    return {
        "statusCode": 200,
        "body": json.dumps({
            "risk_score": risk_score,
            "risk_level": "LOW" if risk_score < 0.5 else "HIGH",
            "model_version": "v2.1",
            "message": f"Risk model invoked (API class: {api_class}, DG approval: {dg_approval})"
        })
    }
`;

const APPROVAL_TOOL_CODE = `
import json

def handler(event, context):
    body = json.loads(event.get("body", "{}")) if isinstance(event.get("body"), str) else event
    amount = body.get("claim_amount", 0)
    risk = body.get("risk_level", "unknown")
    approved = amount <= 500000 and risk in ["low", "medium"]
    return {
        "statusCode": 200,
        "body": json.dumps({
            "decision": "APPROVED" if approved else "ESCALATED",
            "claim_amount": amount,
            "risk_level": risk,
            "message": f"Claim of \${amount:,.0f} with {risk} risk: {'APPROVED' if approved else 'ESCALATED to senior underwriter'}"
        })
    }
`;

console.log("🔧 Creating Lambda execution role...");
const roleArn = await createLambdaRole("AgentCorePolicyLabLambdaRole");
console.log(`   Role ARN: ${roleArn}`);

console.log("\n📦 Deploying Lambda functions...");
const applicationArn = await deployLambda(
  "AgentCore-Policy-ApplicationTool",
  APPLICATION_TOOL_CODE,
  roleArn,
);
const riskModelArn = await deployLambda(
  "AgentCore-Policy-RiskModelTool",
  RISK_MODEL_TOOL_CODE,
  roleArn,
);
const approvalArn = await deployLambda(
  "AgentCore-Policy-ApprovalTool",
  APPROVAL_TOOL_CODE,
  roleArn,
);

console.log("\n✅ Lambda functions deployed:");
console.log(`   ApplicationTool: ${applicationArn}`);
console.log(`   RiskModelTool:   ${riskModelArn}`);
console.log(`   ApprovalTool:    ${approvalArn}`);

In [ ]:
import { GatewayClient } from "../toolkit/mod.ts";
import type { CognitoClientInfo } from "../toolkit/mod.ts";
import { activateOauthClientCredentials } from "../backend/cognito_config.ts";
import {
  type AuthorizerConfiguration,
  type CreateGatewayResponse,
  GetGatewayCommand,
  type GetGatewayResponse,
  ListGatewaysCommand,
} from "@aws-sdk/client-bedrock-agentcore-control";
import {
  CognitoIdentityProviderClient,
  DeleteUserPoolCommand,
  ResourceNotFoundException as CognitoResourceNotFoundException,
} from "@aws-sdk/client-cognito-identity-provider";

const gatewayClient = new GatewayClient({ region });
gatewayClient.logger.setLevel("WARNING");

const GATEWAY_NAME = "InsuranceUnderwritingGateway";
// The Python lab kept this file under backend/policy/; the ported labs keep every generated
// config together in notebooks/environments/, as notebook 09 does.
const CONFIG_FILE = "environments/policy_lab_config.json";

interface LabConfig {
  gateway: {
    gateway_url: string;
    gateway_id: string;
    region: string;
    client_info: CognitoClientInfo;
  };
  policy_engine_id: string | null;
  policy_engine_arn: string | null;
}

/** Python: `json.load(open(CONFIG_FILE))`, guarded by `os.path.exists(CONFIG_FILE)`. */
async function readLabConfig(): Promise<LabConfig | null> {
  try {
    return JSON.parse(await Deno.readTextFile(CONFIG_FILE)) as LabConfig;
  } catch {
    return null;
  }
}

async function saveLabConfig(config: LabConfig): Promise<void> {
  await writeFile(CONFIG_FILE, `${JSON.stringify(config, null, 2)}\n`);
}

async function getGatewaySummaryByName(name: string) {
  const gateways = await gatewayClient.client.send(new ListGatewaysCommand({}));
  return (gateways.items ?? []).find((g) => g.name === name);
}

interface AuthorizerDetails {
  discovery_url?: string;
  allowed_clients: string[];
  user_pool_id?: string;
}

function getGatewayAuthorizerDetails(
  gateway: { authorizerConfiguration?: AuthorizerConfiguration },
): AuthorizerDetails {
  const customJwt = gateway.authorizerConfiguration?.customJWTAuthorizer;
  const discoveryUrl = customJwt?.discoveryUrl;
  const allowedClients = customJwt?.allowedClients ?? [];
  let userPoolId: string | undefined;
  if (discoveryUrl && discoveryUrl.includes("/")) {
    const segments = discoveryUrl.replace(/\/+$/, "").split("/");
    userPoolId = segments[segments.length - 2];
  }
  return {
    discovery_url: discoveryUrl,
    allowed_clients: allowedClients,
    user_pool_id: userPoolId,
  };
}

function gatewayMatchesClientInfo(
  gateway: { authorizerConfiguration?: AuthorizerConfiguration },
  candidateClientInfo: CognitoClientInfo,
): boolean {
  const details = getGatewayAuthorizerDetails(gateway);
  return details.allowed_clients.includes(candidateClientInfo.client_id) &&
    candidateClientInfo.user_pool_id === details.user_pool_id;
}

async function deleteUserPoolIfPresent(userPoolId?: string): Promise<void> {
  if (!userPoolId) return;
  const cognitoClient = new CognitoIdentityProviderClient({ region });
  try {
    await cognitoClient.send(new DeleteUserPoolCommand({ UserPoolId: userPoolId }));
    console.log(`   ✅ Deleted Cognito user pool: ${userPoolId}`);
  } catch (deleteError) {
    if (deleteError instanceof CognitoResourceNotFoundException) return;
    console.log(
      `   ⚠️  Cognito cleanup warning for ${userPoolId}: ${String(deleteError).slice(0, 120)}`,
    );
  }
}

console.log(`🚀 Preparing AgentCore Gateway: ${GATEWAY_NAME}`);

const existingLabConfig = await readLabConfig();
let gateway: GetGatewayResponse | CreateGatewayResponse | null = null;
let clientInfo: CognitoClientInfo | null = null;

if (existingLabConfig) {
  try {
    const existingGatewayId = existingLabConfig.gateway?.gateway_id;
    if (existingGatewayId) {
      gateway = await gatewayClient.client.send(
        new GetGatewayCommand({ gatewayIdentifier: existingGatewayId }),
      );
      clientInfo = existingLabConfig.gateway.client_info;
      if (gatewayMatchesClientInfo(gateway, clientInfo)) {
        await activateOauthClientCredentials(clientInfo, region);
        console.log("   ✅ Reusing existing lab gateway and Cognito configuration");
      } else {
        const details = getGatewayAuthorizerDetails(gateway);
        console.log(
          "   ⚠️  Existing config does not match the gateway authorizer. Recreating lab resources...",
        );
        await gatewayClient.cleanupGateway(existingGatewayId, clientInfo);
        await deleteUserPoolIfPresent(details.user_pool_id);
        await Deno.remove(CONFIG_FILE).catch(() => {});
        gateway = null;
        clientInfo = null;
      }
    }
  } catch (reuseError) {
    console.log(
      `   ⚠️  Existing config could not be reused: ${String(reuseError).slice(0, 120)}`,
    );
    gateway = null;
    clientInfo = null;
    const existingSummary = await getGatewaySummaryByName(GATEWAY_NAME);
    if (existingSummary) {
      try {
        const staleGateway = await gatewayClient.client.send(
          new GetGatewayCommand({ gatewayIdentifier: existingSummary.gatewayId }),
        );
        const staleDetails = getGatewayAuthorizerDetails(staleGateway);
        await gatewayClient.cleanupGateway(
          existingSummary.gatewayId!,
          existingLabConfig.gateway?.client_info ?? null,
        );
        await deleteUserPoolIfPresent(staleDetails.user_pool_id);
      } catch (cleanupError) {
        console.log(
          `   ⚠️  Stale gateway cleanup warning: ${String(cleanupError).slice(0, 120)}`,
        );
      }
    }
    await Deno.remove(CONFIG_FILE).catch(() => {});
  }
}

if (gateway === null) {
  console.log("🔐 Creating inbound OAuth with Cognito for this lab...");
  const cognitoResult = await gatewayClient.createOauthAuthorizerWithCognito(GATEWAY_NAME);
  clientInfo = cognitoResult.client_info;
  await activateOauthClientCredentials(clientInfo, region);
  const authorizerConfig = cognitoResult.authorizer_config;

  if (!authorizerConfig) {
    throw new Error(
      "Missing Cognito authorizer configuration from createOauthAuthorizerWithCognito()",
    );
  }

  gateway = await gatewayClient.createMcpGateway({
    name: GATEWAY_NAME,
    roleArn: null,
    authorizerConfig,
  });
  console.log("   ✅ Gateway created");
}

// `createMcpGateway` answers with the CreateGateway response itself, so the ids are read off it.
const GATEWAY_ID = gateway.gatewayId!;
const GATEWAY_ARN = gateway.gatewayArn!;
const GATEWAY_URL = gateway.gatewayUrl!;
const GATEWAY_ROLE_ARN = gateway.roleArn!;

let labConfig: LabConfig = {
  gateway: {
    gateway_url: GATEWAY_URL,
    gateway_id: GATEWAY_ID,
    region,
    client_info: clientInfo!,
  },
  policy_engine_id: null,
  policy_engine_arn: null,
};

/** Python: the `if os.path.exists(CONFIG_FILE): lab_config = json.load(f)` prologue of a cell. */
async function reloadLabConfig(): Promise<boolean> {
  const loaded = await readLabConfig();
  if (loaded) labConfig = loaded;
  return loaded !== null;
}

await saveLabConfig(labConfig);

console.log(`\n   Gateway ID:  ${GATEWAY_ID}`);
console.log(`   Gateway URL: ${GATEWAY_URL}`);
console.log(`   Config saved to: ${CONFIG_FILE}`);
console.log(`   OAuth client ID: ${clientInfo!.client_id}`);
console.log(
  "   This advanced lab manages its own Cognito setup and can safely reuse its saved config on reruns.",
);

In [ ]:
import {
  ListGatewayTargetsCommand,
  type SchemaDefinition,
  type ToolDefinition,
} from "@aws-sdk/client-bedrock-agentcore-control";
import { PutRolePolicyCommand } from "@aws-sdk/client-iam";
import { roleNameFromArn } from "../toolkit/gateway.ts";

// ── Tool schemas for each Lambda target ───────────────────────────────────
function makeToolSchema(
  toolName: string,
  description: string,
  properties: Record<string, SchemaDefinition>,
  required: string[],
): ToolDefinition {
  return {
    name: toolName,
    description,
    inputSchema: {
      type: "object",
      properties,
      required,
    },
  };
}

const toolSchemas: Record<string, ToolDefinition> = {
  ApplicationToolTarget: makeToolSchema(
    "create_application",
    "Creates an insurance application",
    {
      applicant_region: { type: "string", description: "Geographic region (e.g. US, CA, EU)" },
      coverage_amount: { type: "integer", description: "Requested coverage amount in USD" },
    },
    ["applicant_region", "coverage_amount"],
  ),
  RiskModelToolTarget: makeToolSchema(
    "invoke_risk_model",
    "Invokes the risk scoring model",
    {
      API_classification: {
        type: "string",
        description: "API classification: public, internal, or restricted",
      },
      data_governance_approval: {
        type: "boolean",
        description: "Data governance approval status",
      },
    },
    ["API_classification", "data_governance_approval"],
  ),
  ApprovalToolTarget: makeToolSchema(
    "approve_claim",
    "Approves an insurance underwriting decision",
    {
      claim_amount: { type: "integer", description: "Insurance claim amount in USD" },
      risk_level: { type: "string", description: "Risk level: low, medium, high, or critical" },
    },
    ["claim_amount", "risk_level"],
  ),
};

const targetArns: Record<string, string> = {
  ApplicationToolTarget: applicationArn,
  RiskModelToolTarget: riskModelArn,
  ApprovalToolTarget: approvalArn,
};

console.log("🔗 Attaching Lambda targets to Gateway...");
const listedTargets = await gatewayClient.client.send(
  new ListGatewayTargetsCommand({ gatewayIdentifier: GATEWAY_ID }),
);
const existingTargets = new Set((listedTargets.items ?? []).map((target) => target.name));
const targetFailures: [string, string][] = [];
for (const [targetName, lambdaArn] of Object.entries(targetArns)) {
  if (existingTargets.has(targetName)) {
    console.log(`   ♻️  Reusing existing target: ${targetName}`);
    continue;
  }
  try {
    await gatewayClient.createMcpGatewayTarget({
      gateway: { gatewayId: GATEWAY_ID, roleArn: GATEWAY_ROLE_ARN },
      name: targetName,
      targetType: "lambda",
      targetPayload: {
        lambdaArn,
        toolSchema: { inlinePayload: [toolSchemas[targetName]] },
      },
    });
    console.log(`   ✅ Added target: ${targetName}`);
  } catch (e) {
    const errorMessage = String(e);
    console.log(`   ⚠️  Target ${targetName} failed: ${errorMessage.slice(0, 120)}`);
    targetFailures.push([targetName, errorMessage]);
  }
}

if (targetFailures.length > 0) {
  throw new Error(`Target creation failed: ${JSON.stringify(targetFailures)}`);
}

// A Lambda target is invoked with the Gateway's own execution role, and nothing has granted that
// role `lambda:InvokeFunction` for these functions: the starter toolkit only adds that permission
// for the throwaway Lambda it deploys itself, never for a caller-supplied `lambdaArn`. Without
// this inline policy every tool call comes back as AccessDeniedException.
await iamClient.send(
  new PutRolePolicyCommand({
    RoleName: roleNameFromArn(GATEWAY_ROLE_ARN),
    PolicyName: "GatewayLambdaTargets-AgentCorePolicyLab",
    PolicyDocument: JSON.stringify({
      Version: "2012-10-17",
      Statement: [{
        Sid: "InvokePolicyLabLambdaTargets",
        Effect: "Allow",
        Action: ["lambda:InvokeFunction"],
        Resource: Object.values(targetArns),
      }],
    }),
  }),
);
console.log("   ✅ Gateway execution role allowed to invoke the lab's Lambda targets");

console.log("\n✅ Gateway infrastructure ready!");

---
# Part 3: Agent WITHOUT Policies — Unrestricted Access

Let's first run the agent **without any Policy Engine** attached.
All three tools should be accessible with no restrictions.

> **Key concept**: Without a Policy Engine, the Gateway passes all requests through directly.

In [ ]:
import { Agent, BedrockModel, McpClient } from "@strands-agents/sdk";

const MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0";

/** Get an OAuth access token for the Gateway. */
async function getAccessToken(gatewayCfg: LabConfig["gateway"]): Promise<string> {
  const cfgClientInfo = gatewayCfg.client_info;
  const helperClient = new GatewayClient({ region: gatewayCfg.region ?? region });

  try {
    return await helperClient.getAccessTokenForCognito(cfgClientInfo);
  } catch (helperError) {
    const resp = await fetch(cfgClientInfo.token_endpoint, {
      method: "POST",
      headers: { "Content-Type": "application/x-www-form-urlencoded" },
      body: new URLSearchParams({
        grant_type: "client_credentials",
        client_id: cfgClientInfo.client_id,
        client_secret: cfgClientInfo.client_secret,
        scope: cfgClientInfo.scope ?? "",
      }),
      signal: AbortSignal.timeout(30_000),
    });
    if (resp.status !== 200) {
      throw new Error(
        `Failed to get Cognito access token (${resp.status}): ${
          (await resp.text()).slice(0, 200)
        } | helper error: ${helperError}`,
      );
    }
    const tokenPayload = await resp.json() as { access_token?: string };
    if (!tokenPayload.access_token) {
      throw new Error(`Token response missing access_token: ${JSON.stringify(tokenPayload)}`);
    }
    return tokenPayload.access_token;
  }
}

/** Run a Strands agent connected to the AgentCore Gateway. */
async function runAgentWithTools(
  prompt: string,
  gatewayUrl: string,
  token: string,
): Promise<string> {
  // Python wrapped `streamablehttp_client(url, headers)` in an MCPClient; the TypeScript SDK's
  // McpClient builds that same streamable-HTTP transport itself when it is handed a `url`.
  const mcp = new McpClient({
    url: gatewayUrl,
    headers: { Authorization: `Bearer ${token}` },
  });

  try {
    const tools = await mcp.listTools();
    const toolNames = tools.map((t) => t.name);
    console.log(`   📋 Available tools: ${JSON.stringify(toolNames)}`);

    const model = new BedrockModel({
      modelId: MODEL_ID,
      region,
    });
    const agent = new Agent({
      model,
      tools,
      systemPrompt:
        "You are an insurance underwriting assistant. Use the available tools to help with insurance tasks.",
    });
    const response = await agent.invoke(prompt);
    return String(response);
  } finally {
    await mcp.disconnect();
  }
}

let accessToken = "";

if (await reloadLabConfig()) {
  accessToken = await getAccessToken(labConfig.gateway);

  console.log("🤖 Running agent WITHOUT policies (unrestricted)...\n");
  console.log("Test 1: List tools");
  const result = await runAgentWithTools(
    "What tools do you have available?",
    labConfig.gateway.gateway_url,
    accessToken,
  );
  console.log(`   Response: ${result.slice(0, 300)}`);
} else {
  console.log("ℹ️  Run Part 2 first to create the Gateway and OAuth configuration.");
}

In [ ]:
// Test all three tools — no restrictions yet!
if (await reloadLabConfig()) {
  accessToken = await getAccessToken(labConfig.gateway);
  const gwUrl = labConfig.gateway.gateway_url;

  console.log("🧪 Testing all tools WITHOUT policy restrictions...\n");

  console.log("Test 2: Large coverage application ($5M — would be blocked by policy later)");
  const r2 = await runAgentWithTools(
    "Create an application for US region with $5 million coverage.",
    gwUrl,
    accessToken,
  );
  console.log(`   Result: ${r2.slice(0, 200)}\n`);

  console.log("Test 3: Risk model invocation");
  const r3 = await runAgentWithTools(
    "Invoke the risk model with public API classification and data governance approval set to true.",
    gwUrl,
    accessToken,
  );
  console.log(`   Result: ${r3.slice(0, 200)}\n`);

  console.log("Test 4: Claim approval");
  const r4 = await runAgentWithTools(
    "Approve a claim for $75,000 with medium risk level.",
    gwUrl,
    accessToken,
  );
  console.log(`   Result: ${r4.slice(0, 200)}\n`);

  console.log("\n💡 OBSERVATION: Without policies, ALL tools are accessible.");
  console.log("   The agent can create ANY coverage amount — no limits.");
  console.log("   This is the problem we're about to solve with Cedar policies.");
}

---
# Part 4: Create and Attach a Policy Engine

Now let's add a **Policy Engine** to the Gateway.

**Key concept**: Once a Policy Engine is attached:
- **Default Deny** kicks in immediately
- An empty Policy Engine = **ALL tools blocked**
- You must explicitly write `permit` policies to allow tool access

In [ ]:
import { PolicyClient } from "../toolkit/mod.ts";
import { ListPolicyEnginesCommand } from "@aws-sdk/client-bedrock-agentcore-control";

const policyAdminClient = new PolicyClient({ region });
// Python: `boto3.client("bedrock-agentcore-control")`. The helper keeps that raw client public.
const agentcoreCtrl = policyAdminClient.client;

console.log("🔧 Creating Policy Engine...");

const engines = (await agentcoreCtrl.send(new ListPolicyEnginesCommand({}))).policyEngines ?? [];
const existingEngine = engines.find((e) => e.name === "InsuranceUnderwritingPolicyEngine");
if (existingEngine) {
  console.log(
    "   ♻️  Existing Policy Engine found — cleaning it up for a fresh rerun",
  );
  try {
    // Detaching means dropping the whole policyEngineConfiguration block from the UpdateGateway
    // call, which is what `policyEngineConfig: null` does. (The Python lab rebuilt the request by
    // hand to achieve the same thing.)
    await gatewayClient.updateGateway({
      gatewayIdentifier: GATEWAY_ID,
      policyEngineConfig: null,
    });
    console.log("   ✅ Detached existing Policy Engine from Gateway");
  } catch (detachError) {
    console.log(`   ⚠️  Gateway detach warning: ${String(detachError).slice(0, 120)}`);
  }
  await policyAdminClient.cleanupPolicyEngine(existingEngine.policyEngineId!);
  let engineRemoved = false;
  for (let attempt = 0; attempt < 30; attempt++) {
    const remainingEngines =
      (await agentcoreCtrl.send(new ListPolicyEnginesCommand({}))).policyEngines ?? [];
    if (!remainingEngines.some((engine) => engine.name === "InsuranceUnderwritingPolicyEngine")) {
      console.log("   ✅ Previous Policy Engine fully removed");
      engineRemoved = true;
      break;
    }
    await sleep(2_000);
  }
  if (!engineRemoved) {
    throw new Error("Timed out waiting for the previous Policy Engine to be deleted");
  }
}

const engine = await policyAdminClient.createPolicyEngine({
  name: "InsuranceUnderwritingPolicyEngine",
  description: "Policy engine controlling insurance underwriting agent tool access",
  tags: { Environment: "Lab", Module: "AgentCorePolicy" },
});
console.log("   ✅ Policy Engine Created");

const POLICY_ENGINE_ID = engine.policyEngineId!;
const POLICY_ENGINE_ARN = engine.policyEngineArn!;
console.log(`   ID:  ${POLICY_ENGINE_ID}`);
console.log(`   ARN: ${POLICY_ENGINE_ARN}`);

if (await reloadLabConfig()) {
  labConfig.policy_engine_id = POLICY_ENGINE_ID;
  labConfig.policy_engine_arn = POLICY_ENGINE_ARN;
  await saveLabConfig(labConfig);
  await state.set("policy_lab", labConfig as never);
}

In [ ]:
const gwClient = new GatewayClient({ region });

console.log("🔗 Attaching Policy Engine to Gateway in ENFORCE mode...");
console.log("   ⚠️  After this: ALL tools will be BLOCKED (default-deny!)\n");

let engineActive = false;
for (let attempt = 0; attempt < 30; attempt++) {
  const currentEngine = await policyAdminClient.getPolicyEngine(POLICY_ENGINE_ID);
  if (currentEngine.status === "ACTIVE") {
    engineActive = true;
    break;
  }
  await sleep(2_000);
}
if (!engineActive) {
  throw new Error(`Policy Engine ${POLICY_ENGINE_ID} did not become ACTIVE in time`);
}

await gwClient.updateGatewayPolicyEngine({
  gatewayIdentifier: GATEWAY_ID,
  policyEngineArn: POLICY_ENGINE_ARN,
  mode: "ENFORCE",
});

console.log("✅ Policy Engine attached in ENFORCE mode");
console.log("");
console.log("Now let's see what happens when the agent tries to list tools...");

In [ ]:
// Demonstrate DEFAULT DENY — empty Policy Engine blocks everything
interface RawToolListResponse {
  result?: { tools?: { name?: string }[] };
}

/** Call the Gateway tool list endpoint directly. */
async function listAvailableToolsRaw(
  gatewayUrl: string,
  token: string,
): Promise<RawToolListResponse> {
  const resp = await fetch(gatewayUrl, {
    method: "POST",
    headers: {
      Authorization: `Bearer ${token}`,
      "Content-Type": "application/json",
    },
    body: JSON.stringify({
      jsonrpc: "2.0",
      id: "list-tools",
      method: "tools/list",
      params: {},
    }),
    signal: AbortSignal.timeout(15_000),
  });
  return await resp.json() as RawToolListResponse;
}

if (await reloadLabConfig()) {
  accessToken = await getAccessToken(labConfig.gateway);

  console.log("🔍 Listing available tools (with EMPTY Policy Engine attached)...\n");
  const toolList = await listAvailableToolsRaw(labConfig.gateway.gateway_url, accessToken);
  console.log("Raw Gateway Response:");
  console.log(JSON.stringify(toolList, null, 2));

  const tools = toolList.result?.tools ?? [];
  console.log(`\n📋 Number of available tools: ${tools.length}`);

  if (tools.length === 0) {
    console.log("\n✅ DEFAULT DENY CONFIRMED!");
    console.log("   The empty Policy Engine is blocking ALL tool discovery.");
    console.log("   Agents cannot see OR call any tools.");
    console.log("   This is the safe-fail posture.");
  } else {
    console.log(`\n   Tools visible: ${JSON.stringify(tools.map((t) => t.name))}`);
  }
}

---
# Part 5: Write Cedar Policies

Now let's write Cedar policies to selectively allow tool access.

**Our Rules:**
1. Allow `create_application` only if `coverage_amount ≤ $1,000,000`
2. Allow `invoke_risk_model` only if `data_governance_approval == true`
3. Allow `approve_claim` for all users (no conditions)

**Cedar Action name format:** `TargetName___operation_name`

In [ ]:
const policyClient = new PolicyClient({ region });

console.log("📝 Creating Cedar policies...\n");

// ── Policy 1: Application creation — coverage_amount ≤ $1M ────────────────
const cedarP1 = "permit(principal, " +
  `action == AgentCore::Action::"ApplicationToolTarget___create_application", ` +
  `resource == AgentCore::Gateway::"${GATEWAY_ARN}") ` +
  "when { context.input.coverage_amount <= 1000000 };";

await policyClient.createOrGetPolicy({
  policyEngineId: POLICY_ENGINE_ID,
  name: "policy_create_application",
  description: "Allow application creation for coverage ≤ $1M",
  definition: { cedar: { statement: cedarP1 } },
});
console.log("   ✅ Policy 1: Application Tool (coverage ≤ $1M)");
console.log(`      Cedar: ${cedarP1.slice(0, 100)}...`);

// ── Policy 2: Risk model — requires data governance approval ──────────────
const cedarP2 = "permit(principal, " +
  `action == AgentCore::Action::"RiskModelToolTarget___invoke_risk_model", ` +
  `resource == AgentCore::Gateway::"${GATEWAY_ARN}") ` +
  "when { context.input.data_governance_approval == true };";

await policyClient.createOrGetPolicy({
  policyEngineId: POLICY_ENGINE_ID,
  name: "policy_risk_model",
  description: "Allow risk model only with data governance approval",
  definition: { cedar: { statement: cedarP2 } },
});
console.log("\n   ✅ Policy 2: Risk Model Tool (data governance required)");

// ── Policy 3: Approval tool — open access ─────────────────────────────────
const cedarP3 = "permit(principal, " +
  `action == AgentCore::Action::"ApprovalToolTarget___approve_claim", ` +
  `resource == AgentCore::Gateway::"${GATEWAY_ARN}");`;

await policyClient.createOrGetPolicy({
  policyEngineId: POLICY_ENGINE_ID,
  name: "policy_approve_claim",
  description: "Allow claim approval for all users",
  validationMode: "IGNORE_ALL_FINDINGS",
  definition: { cedar: { statement: cedarP3 } },
});
console.log("\n   ✅ Policy 3: Approval Tool (open access)");

console.log("\n🎉 All Cedar policies created!");
console.log("   Tools should now be discoverable and callable (within policy limits).");

---
# Part 6: Test Policy Enforcement

Now let's test each Cedar rule with explicit allow and deny scenarios.

We expect:
- ✅ $750K coverage → **ALLOWED** (below $1M limit)
- ❌ $1.5M coverage → **DENIED** (exceeds $1M limit)
- ✅ Risk model with approval=true → **ALLOWED**
- ❌ Risk model with approval=false → **DENIED**


In [ ]:
console.log("=".repeat(60));
console.log("TEST 1: ALLOW Scenario — $750K Coverage (≤ $1M limit)");
console.log("=".repeat(60));
console.log("Expected: Cedar evaluates 750000 <= 1000000 → TRUE → ALLOW");
console.log();

if (await reloadLabConfig()) {
  accessToken = await getAccessToken(labConfig.gateway);

  const result = await runAgentWithTools(
    "Create an application for US region with $750,000 coverage.",
    labConfig.gateway.gateway_url,
    accessToken,
  );
  console.log(`Agent Response: ${result}`);
  console.log();
  console.log(
    "✅ If you see a successful application creation above → Policy ALLOWED it!",
  );
}

In [ ]:
console.log("=".repeat(60));
console.log("TEST 2: DENY Scenario — $1.5M Coverage (> $1M limit)");
console.log("=".repeat(60));
console.log("Expected: Cedar evaluates 1500000 <= 1000000 → FALSE → DENY");
console.log();

if (await reloadLabConfig()) {
  accessToken = await getAccessToken(labConfig.gateway);

  const result = await runAgentWithTools(
    "Create an application for US region with $1.5 million coverage.",
    labConfig.gateway.gateway_url,
    accessToken,
  );
  console.log(`Agent Response: ${result}`);
  console.log();
  console.log("❌ The agent should report that the action was denied or not possible.");
  console.log("   The Lambda was NEVER called — Cedar blocked it at the Gateway.");
}

In [ ]:
console.log("=".repeat(60));
console.log("TEST 3: ALLOW Scenario — Risk Model With Approval");
console.log("=".repeat(60));
console.log("Expected: Cedar evaluates data_governance_approval == true → ALLOW");
console.log();

if (await reloadLabConfig()) {
  accessToken = await getAccessToken(labConfig.gateway);

  const result = await runAgentWithTools(
    "Invoke the risk model with public API classification and data governance approval set to true.",
    labConfig.gateway.gateway_url,
    accessToken,
  );
  console.log(`Agent Response: ${result}`);
  console.log();
  console.log("✅ If you see a successful risk-model response above → Policy ALLOWED it!");
}

In [ ]:
console.log("=".repeat(60));
console.log("TEST 4: DENY Scenario — Risk Model Without Approval");
console.log("=".repeat(60));
console.log("Expected: Cedar evaluates data_governance_approval == false → DENY");
console.log();

if (await reloadLabConfig()) {
  accessToken = await getAccessToken(labConfig.gateway);

  const result = await runAgentWithTools(
    "Invoke the risk model with internal API classification and data governance approval set to false.",
    labConfig.gateway.gateway_url,
    accessToken,
  );
  console.log(`Agent Response: ${result}`);
  console.log();
  console.log(
    "❌ The agent should report that the risk model invocation was denied or not possible.",
  );
  console.log("   The Lambda should not be invoked when Cedar blocks the request.");
}

---
# Part 7: NL2Cedar — Natural Language Policy Authoring

Now let's generate Cedar policies from **plain English** using the Policy Authoring Service.

This is the feature that lets security teams write policies without learning Cedar syntax.

In [ ]:
import { cedarStatement } from "../toolkit/policy.ts";

console.log("📝 NL2Cedar: Multi-line input → multiple policies\n");

const nlMulti =
  `Allow all users to invoke the risk model tool when data governance approval is true.`;

console.log(`Natural Language Input:\n${nlMulti}\n`);

const resultMulti = await policyClient.generatePolicy({
  policyEngineId: POLICY_ENGINE_ID,
  name: `nl_multi_${Math.floor(Date.now() / 1000)}`,
  resource: { arn: GATEWAY_ARN },
  content: { rawText: nlMulti },
  fetchAssets: true,
});

if (resultMulti.generatedPolicies && resultMulti.generatedPolicies.length > 0) {
  console.log(`Generated ${resultMulti.generatedPolicies.length} policy assets:\n`);
  for (const [index, gp] of resultMulti.generatedPolicies.entries()) {
    const cedar = cedarStatement(gp.definition);
    console.log(`Policy ${index + 1}:`);
    if (cedar) {
      console.log(cedar);
    } else {
      console.log(JSON.stringify(gp, null, 2));
    }
    console.log();
  }
  console.log("✅ Multi-line NL2Cedar generation completed.");
}

---
# Part 8: 🧹 Cleanup

Clean up all resources created in this lab.

**Order of operations (important!):**
1. Detach Policy Engine from Gateway
2. Delete policies from Policy Engine
3. Delete the Policy Engine
4. Delete the Gateway (and targets)
5. Delete Lambda functions and IAM role
6. Delete the lab Cognito user pool and local config file

In [ ]:
// Run this cell to clean up all lab resources
import { DeleteFunctionCommand } from "@aws-sdk/client-lambda";
import {
  DeleteRoleCommand,
  DeleteRolePolicyCommand,
  DetachRolePolicyCommand,
} from "@aws-sdk/client-iam";

console.log("🧹 Starting cleanup...\n");

// ── Step 1: Detach Policy Engine from Gateway ─────────────────────────────
try {
  console.log("Step 1: Detaching Policy Engine from Gateway...");
  await gwClient.updateGatewayPolicyEngine({
    gatewayIdentifier: GATEWAY_ID,
    policyEngineArn: null,
    mode: null,
  });
  console.log("   ✅ Policy Engine detached");
} catch (e) {
  console.log(`   ⚠️  ${String(e).slice(0, 80)}`);
}

// ── Step 2 & 3: Delete policies then Policy Engine ────────────────────────
try {
  console.log("\nStep 2: Cleaning up Policy Engine (all policies)...");
  await policyClient.cleanupPolicyEngine(POLICY_ENGINE_ID);
  console.log("   ✅ Policy Engine and all policies deleted");
} catch (e) {
  console.log(`   ⚠️  ${String(e).slice(0, 80)}`);
}

// ── Step 4: Delete Gateway ────────────────────────────────────────────────
if (await reloadLabConfig()) {
  try {
    console.log("\nStep 3: Cleaning up Gateway...");
    await gwClient.cleanupGateway(
      labConfig.gateway.gateway_id,
      labConfig.gateway.client_info,
    );
    console.log("   ✅ Gateway deleted");
  } catch (e) {
    console.log(`   ⚠️  ${String(e).slice(0, 80)}`);
  }
}

// ── Step 5: Delete Lambda functions ──────────────────────────────────────
console.log("\nStep 4: Deleting Lambda functions...");
for (
  const fnName of [
    "AgentCore-Policy-ApplicationTool",
    "AgentCore-Policy-RiskModelTool",
    "AgentCore-Policy-ApprovalTool",
  ]
) {
  try {
    await lambdaClient.send(new DeleteFunctionCommand({ FunctionName: fnName }));
    console.log(`   ✅ Deleted: ${fnName}`);
  } catch (e) {
    console.log(`   ⚠️  ${fnName}: ${String(e).slice(0, 60)}`);
  }
}

// ── Step 6: Delete IAM role ───────────────────────────────────────────────
console.log("\nStep 5: Cleaning up IAM role...");
try {
  await iamClient.send(
    new DetachRolePolicyCommand({
      RoleName: "AgentCorePolicyLabLambdaRole",
      PolicyArn: "arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    }),
  );
  await iamClient.send(new DeleteRoleCommand({ RoleName: "AgentCorePolicyLabLambdaRole" }));
  console.log("   ✅ IAM role deleted");
} catch (e) {
  console.log(`   ⚠️  ${String(e).slice(0, 80)}`);
}

// The Gateway execution role outlives this lab (it is shared by every gateway the course makes),
// so only the inline policy this notebook added to it is removed.
try {
  await iamClient.send(
    new DeleteRolePolicyCommand({
      RoleName: roleNameFromArn(GATEWAY_ROLE_ARN),
      PolicyName: "GatewayLambdaTargets-AgentCorePolicyLab",
    }),
  );
  console.log("   ✅ Gateway Lambda invoke policy removed");
} catch (e) {
  console.log(`   ⚠️  ${String(e).slice(0, 80)}`);
}

// ── Step 7: Delete Cognito lab resources and local config ─────────────────
console.log("\nStep 6: Cleaning up Cognito lab resources...");
if (await reloadLabConfig()) {
  try {
    const cognitoClient = new CognitoIdentityProviderClient({
      region: labConfig.gateway.region ?? region,
    });
    await cognitoClient.send(
      new DeleteUserPoolCommand({ UserPoolId: labConfig.gateway.client_info.user_pool_id }),
    );
    console.log(
      `   ✅ Deleted Cognito user pool: ${labConfig.gateway.client_info.user_pool_id}`,
    );
  } catch (e) {
    console.log(`   ⚠️  Cognito cleanup: ${String(e).slice(0, 80)}`);
  }

  try {
    await Deno.remove(CONFIG_FILE);
    console.log(`   ✅ Removed local config: ${CONFIG_FILE}`);
  } catch (e) {
    console.log(`   ⚠️  Local config cleanup: ${String(e).slice(0, 80)}`);
  }
}

console.log("\n🎉 Cleanup complete!");

---
# 🎉 Lab Complete!

## What You Accomplished

| Step | Concept | Result |
|------|---------|--------|
| Part 2 | Gateway + Lambda targets | Infrastructure deployed |
| Part 3 | No Policy Engine | All tools accessible (no limits) |
| Part 4 | Policy Engine attached | Default Deny — all tools blocked |
| Part 5 | Cedar policies created | Specific tools selectively unlocked |
| Part 6 | ALLOW test ($750K) | Policy permits — application created |
| Part 6 | DENY test ($1.5M) | Policy blocks — Lambda never called |
| Part 7 | NL2Cedar | Natural language → Cedar policies |
| Part 8 | Emergency shutdown | One forbid line blocks everything |

## Key Takeaways

- 🛡️ **Cedar policies operate outside the LLM** — prompt injection cannot bypass them
- 🚫 **Default Deny**: empty Policy Engine = all tools blocked
- ⚡ **Emergency shutdown** = one line of Cedar: `forbid(principal, action, resource);`
- 🗣️ **NL2Cedar** converts plain English to valid Cedar — no syntax expertise needed
- 🎯 **Test first with LOG_ONLY** before switching to ENFORCE in production